In [50]:

import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")


In [ ]:

# ── CONFIGURATION
REF_DATE      = pd.Timestamp("2026-03-05")
EXCEL_FILE    = "rfm_output/ShopNest_Capstone_Dataset.xlsx"
OUTPUT_DIR    = "rfm_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

SEG_COLORS = {
    "Champions":          "#1F4E79",
    "Loyal Customers":    "#2E75B6",
    "Potential Loyalists":"#4472C4",
    "New Customers":      "#70AD47",
    "At Risk":            "#FF7F00",
    "Hibernating":        "#843C0C",
    "Lost Customers":     "#C00000",
    "Need Attention":     "#7030A0",
}

In [52]:


# ── 1. LOAD DATA
print("=" * 65)
print("  ShopNest RFM Capstone Analysis")
print("=" * 65)
print("\n[1] Loading data …")

try:
    df_cust  = pd.read_excel(EXCEL_FILE, sheet_name="Customers")
    df_ord   = pd.read_excel(EXCEL_FILE, sheet_name="Orders")
    df_items = pd.read_excel(EXCEL_FILE, sheet_name="Order_Items")
    print(f"    Customers  : {len(df_cust):>7,}")
    print(f"    Orders     : {len(df_ord):>7,}")
    print(f"    Order Items: {len(df_items):>7,}")
except FileNotFoundError:
    # fallback to CSVs
    print("    Excel not found — trying CSVs …")
    df_cust  = pd.read_csv("customers.csv")
    df_ord   = pd.read_csv("orders.csv")
    df_items = pd.read_csv("order_items.csv")

# normalise column names
df_cust.columns  = df_cust.columns.str.strip().str.lower().str.replace(" ","_")
df_ord.columns   = df_ord.columns.str.strip().str.lower().str.replace(" ","_")
df_items.columns = df_items.columns.str.strip().str.lower().str.replace(" ","_")

df_ord["order_date"] = pd.to_datetime(df_ord["order_date"])


  ShopNest RFM Capstone Analysis

[1] Loading data …
    Customers  :   5,000
    Orders     :  35,433
    Order Items: 106,289


In [53]:

print("\n[2] Data quality check …")
completed = df_ord[df_ord["order_status"] == "completed"].copy()
print(f"    Completed orders : {len(completed):,}  "
      f"({len(completed)/len(df_ord)*100:.1f}% of total)")
print(f"    Returned  orders : {(df_ord['order_status']=='returned').sum():,}")
print(f"    Cancelled orders : {(df_ord['order_status']=='cancelled').sum():,}")
print(f"    Null order_total : {df_ord['order_total'].isna().sum()}")
print(f"    Date range       : {df_ord['order_date'].min().date()} → "
      f"{df_ord['order_date'].max().date()}")


[2] Data quality check …
    Completed orders : 23,695  (66.9% of total)
    Returned  orders : 5,857
    Cancelled orders : 5,881
    Null order_total : 0
    Date range       : 2020-01-01 → 2024-12-30


In [54]:


print("\n[3] Computing RFM metrics …")

rfm = completed.groupby("customer_id").agg(
    last_order_date = ("order_date",   "max"),
    frequency       = ("order_id",     "nunique"),
    monetary        = ("order_total",  "sum"),
    avg_order_value = ("order_total",  "mean"),
    n_returns       = ("order_id",     "count"),   # proxy
).reset_index()

rfm["recency_days"] = (REF_DATE - rfm["last_order_date"]).dt.days
rfm["monetary"]     = rfm["monetary"].round(2)
rfm["avg_order_value"] = rfm["avg_order_value"].round(2)

print(f"    Customers analysed : {len(rfm):,}")
print(f"    Avg recency (days) : {rfm['recency_days'].mean():.1f}")
print(f"    Avg frequency      : {rfm['frequency'].mean():.1f}")
print(f"    Avg monetary ($)   : ${rfm['monetary'].mean():,.2f}")
print(f"    Total revenue ($)  : ${rfm['monetary'].sum():,.2f}")


[3] Computing RFM metrics …
    Customers analysed : 4,772
    Avg recency (days) : 438.1
    Avg frequency      : 5.0
    Avg monetary ($)   : $6,730.47
    Total revenue ($)  : $32,117,809.72


In [55]:

print("\n[4] Applying NTILE(5) scores …")

def ntile5(series, ascending=True):
    ranks = series.rank(method="first", ascending=ascending)
    n = len(ranks)
    return ((ranks - 1) * 5 // n + 1).clip(1, 5).astype(int)

rfm["r_score"] = ntile5(rfm["recency_days"], ascending=False)
rfm["f_score"] = ntile5(rfm["frequency"],    ascending=True)
rfm["m_score"] = ntile5(rfm["monetary"],     ascending=True)
rfm["rfm_score"] = (rfm["r_score"]*0.4 + rfm["f_score"]*0.3 + rfm["m_score"]*0.3).round(2)

print("    Score distribution:")
for s in ["r_score","f_score","m_score"]:
    dist = rfm[s].value_counts().sort_index()
    print(f"      {s}: " + " | ".join(f"{k}:{v:,}" for k,v in dist.items()))


[4] Applying NTILE(5) scores …
    Score distribution:
      r_score: 1:955 | 2:954 | 3:955 | 4:954 | 5:954
      f_score: 1:955 | 2:954 | 3:955 | 4:954 | 5:954
      m_score: 1:955 | 2:954 | 3:955 | 4:954 | 5:954


In [56]:


print("\n[5] Assigning customer segments …")

def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4: return "Champions"
    if r >= 4 and f == 1:            return "New Customers"
    if r >= 3 and f >= 3 and m >= 3: return "Loyal Customers"
    if r >= 4 and f <= 3:            return "Potential Loyalists"
    if r <= 2 and f >= 3:            return "At Risk"
    if r <= 2 and f <= 2 and m <= 2: return "Lost Customers"
    if r <= 2:                       return "Hibernating"
    return "Need Attention"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm["segment"].value_counts()
print("    Segment distribution:")
for seg, cnt in seg_counts.items():
    pct = cnt / len(rfm) * 100
    bar = "█" * int(pct / 2)
    print(f"      {seg:<22}: {cnt:>5,} ({pct:5.1f}%)  {bar}")


[5] Assigning customer segments …
    Segment distribution:
      Champions             : 1,078 ( 22.6%)  ███████████
      Lost Customers        :   980 ( 20.5%)  ██████████
      Loyal Customers       :   888 ( 18.6%)  █████████
      At Risk               :   645 ( 13.5%)  ██████
      Need Attention        :   459 (  9.6%)  ████
      Potential Loyalists   :   295 (  6.2%)  ███
      Hibernating           :   284 (  6.0%)  ██
      New Customers         :   143 (  3.0%)  █


In [57]:
print("\n[6] Merging with customer profiles …")
merge_cols = ["customer_id","first_name","last_name","city","country",
              "loyalty_tier","acquisition_channel"]
available  = [c for c in merge_cols if c in df_cust.columns]
rfm_full   = rfm.merge(df_cust[available], on="customer_id", how="left")
print(f"    Final RFM dataset shape: {rfm_full.shape}")



[6] Merging with customer profiles …
    Final RFM dataset shape: (4772, 18)


In [58]:

print("\n[7] Segment business summary:")
seg_summary = rfm_full.groupby("segment").agg(
    customers        = ("customer_id",   "count"),
    avg_recency      = ("recency_days",  "mean"),
    avg_frequency    = ("frequency",     "mean"),
    avg_monetary     = ("monetary",      "mean"),
    total_revenue    = ("monetary",      "sum"),
    avg_rfm_score    = ("rfm_score",     "mean"),
).round(2).reset_index()
seg_summary["pct_base"]  = (seg_summary["customers"] / len(rfm_full) * 100).round(1)
seg_summary["rev_share"] = (seg_summary["total_revenue"] / seg_summary["total_revenue"].sum() * 100).round(1)
seg_summary = seg_summary.sort_values("avg_rfm_score", ascending=False)

print(f"\n  {'Segment':<22} {'Custs':>6} {'%Base':>6} {'AvgRec':>7} "
      f"{'AvgFreq':>8} {'AvgSpend':>10} {'TotalRev':>14} {'RevShare':>9}")
print("  " + "-"*85)
for _, r in seg_summary.iterrows():
    print(f"  {r['segment']:<22} {r['customers']:>6,} {r['pct_base']:>5.1f}% "
          f"{r['avg_recency']:>7.1f} {r['avg_frequency']:>8.1f} "
          f"${r['avg_monetary']:>9,.0f} ${r['total_revenue']:>13,.0f} "
          f"{r['rev_share']:>8.1f}%")
total_rev = seg_summary["total_revenue"].sum()
print("  " + "-"*85)
print(f"  {'TOTAL':<22} {len(rfm_full):>6,} {'100.0':>6}%"
      f"{'':>28}${total_rev:>13,.0f} {'100.0':>8}%")


[7] Segment business summary:

  Segment                 Custs  %Base  AvgRec  AvgFreq   AvgSpend       TotalRev  RevShare
  -------------------------------------------------------------------------------------
  Champions               1,078  22.6%   112.1     10.7 $   15,399 $   16,599,793     51.7%
  Loyal Customers           888  18.6%   247.9      5.8 $    7,984 $    7,090,230     22.1%
  Potential Loyalists       295   6.2%   142.7      2.6 $    2,550 $      752,260      2.3%
  At Risk                   645  13.5%   666.8      4.5 $    5,761 $    3,715,565     11.6%
  New Customers             143   3.0%   139.0      1.2 $    1,472 $      210,476      0.7%
  Need Attention            459   9.6%   309.3      2.7 $    2,400 $    1,101,407      3.4%
  Hibernating               284   6.0%   831.1      1.9 $    5,405 $    1,535,067      4.8%
  Lost Customers            980  20.5%   897.5      1.5 $    1,136 $    1,113,011      3.5%
  --------------------------------------------------

In [59]:
print("\n[8] Computing churn risk scores …")
rfm_full["churn_risk"] = (
    ntile5(rfm_full["recency_days"], ascending=True)  * 0.5 +
    (6 - rfm_full["f_score"])                         * 0.3 +
    (6 - rfm_full["m_score"])                         * 0.2
).round(2)

rfm_full["churn_label"] = pd.cut(
    rfm_full["churn_risk"],
    bins=[0, 2.5, 3.5, 5.1],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)
churn_dist = rfm_full["churn_label"].value_counts()
print(f"    High Risk   : {churn_dist.get('High Risk', 0):,} customers")
print(f"    Medium Risk : {churn_dist.get('Medium Risk', 0):,} customers")
print(f"    Low Risk    : {churn_dist.get('Low Risk', 0):,} customers")



[8] Computing churn risk scores …
    High Risk   : 1,735 customers
    Medium Risk : 1,135 customers
    Low Risk    : 1,902 customers


In [60]:
print("\n[9] Monthly revenue trend …")
monthly = completed.copy()
monthly["year_month"] = monthly["order_date"].dt.to_period("M")
monthly_rev = monthly.groupby("year_month").agg(
    orders  = ("order_id",    "count"),
    revenue = ("order_total", "sum"),
).reset_index()
monthly_rev["revenue"] = monthly_rev["revenue"].round(2)
monthly_rev["mom_pct"] = monthly_rev["revenue"].pct_change().mul(100).round(1)
print(f"    Months covered : {len(monthly_rev)}")
print(f"    Peak month     : {monthly_rev.loc[monthly_rev['revenue'].idxmax(), 'year_month']} "
      f"(${monthly_rev['revenue'].max():,.0f})")
print(f"    Avg monthly rev: ${monthly_rev['revenue'].mean():,.0f}")


[9] Monthly revenue trend …
    Months covered : 60
    Peak month     : 2024-03 ($1,072,795)
    Avg monthly rev: $535,297


In [61]:
# CATEGORY ANALYSIS
print("\n[10] Category performance …")
cat_items = df_items.merge(df_ord[["order_id","order_status"]], on="order_id")
cat_items = cat_items[cat_items["order_status"] == "completed"]
cat_col   = "category" if "category" in cat_items.columns else "Category"
lt_col    = "line_total" if "line_total" in cat_items.columns else "Line Total ($)"
cat_perf  = cat_items.groupby(cat_col).agg(
    revenue = (lt_col, "sum"),
    units   = ("quantity" if "quantity" in cat_items.columns else "Quantity", "sum"),
).sort_values("revenue", ascending=False).round(2)
print("    Top 5 categories by revenue:")
for cat, row in cat_perf.head(5).iterrows():
    pct = row["revenue"] / cat_perf["revenue"].sum() * 100
    print(f"      {cat:<25}: ${row['revenue']:>12,.0f}  ({pct:.1f}%)")


[10] Category performance …
    Top 5 categories by revenue:
      Electronics              : $  21,503,598  (67.0%)
      Home & Office            : $   2,723,827  (8.5%)
      Apparel                  : $   2,176,402  (6.8%)
      Home & Kitchen           : $   1,975,455  (6.2%)
      Accessories              : $   1,790,484  (5.6%)


In [62]:
# SAVE OUTPUTS
print(f"\n[11] Saving results to /{OUTPUT_DIR}/ …")

rfm_full.drop(columns=["last_order_date"], errors="ignore").to_csv(
    f"{OUTPUT_DIR}/rfm_results.csv", index=False)
seg_summary.to_csv(f"{OUTPUT_DIR}/segment_summary.csv", index=False)
monthly_rev.to_csv(f"{OUTPUT_DIR}/monthly_revenue.csv", index=False)
cat_perf.reset_index().to_csv(f"{OUTPUT_DIR}/category_performance.csv", index=False)
print(f"    rfm_results.csv       ({len(rfm_full):,} rows)")
print(f"    segment_summary.csv   ({len(seg_summary)} rows)")
print(f"    monthly_revenue.csv   ({len(monthly_rev)} rows)")
print(f"    category_performance.csv")



[11] Saving results to /rfm_output/ …
    rfm_results.csv       (4,772 rows)
    segment_summary.csv   (8 rows)
    monthly_revenue.csv   (60 rows)
    category_performance.csv


In [63]:
#  PLOTS
print("\n[12] Generating charts …")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns

    plt.rcParams.update({
        "font.family": "DejaVu Sans", "axes.spines.top": False,
        "axes.spines.right": False, "figure.dpi": 150,
    })

    seg_order = [s for s in SEG_COLORS if s in rfm_full["segment"].unique()]

    # ── Chart 1: Customer count per segment (horizontal bar)
    fig, ax = plt.subplots(figsize=(10, 6))
    seg_cnt = rfm_full["segment"].value_counts().reindex(seg_order).dropna()
    colors  = [SEG_COLORS[s] for s in seg_cnt.index]
    bars = ax.barh(seg_cnt.index, seg_cnt.values, color=colors, edgecolor="white", height=0.7)
    for bar, val in zip(bars, seg_cnt.values):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", fontsize=9)
    ax.set_xlabel("Number of Customers", fontsize=10)
    ax.set_title("Customer Distribution by RFM Segment", fontsize=13, fontweight="bold", pad=15)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/01_segment_customer_count.png", bbox_inches="tight")
    plt.close()
    print("    01_segment_customer_count.png")

    # ── Chart 2: Revenue share per segment (donut)
    fig, ax = plt.subplots(figsize=(9, 9))
    rev_by_seg = rfm_full.groupby("segment")["monetary"].sum().reindex(seg_order).dropna()
    colors = [SEG_COLORS[s] for s in rev_by_seg.index]
    wedges, texts, autotexts = ax.pie(
        rev_by_seg.values, labels=None, autopct="%1.1f%%",
        colors=colors, startangle=140, pctdistance=0.80,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2)
    )
    for at in autotexts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    legend_patches = [mpatches.Patch(color=SEG_COLORS[s], label=f"{s}  (${rev_by_seg[s]/1e6:.1f}M)")
                      for s in rev_by_seg.index]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=8.5, frameon=False)
    ax.set_title("Revenue Share by Customer Segment", fontsize=13, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/02_revenue_share_donut.png", bbox_inches="tight")
    plt.close()
    print("    02_revenue_share_donut.png")

    # ── Chart 3: RFM Score distribution (violin)
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_data = rfm_full[rfm_full["segment"].isin(seg_order)].copy()
    seg_rfm   = [plot_data[plot_data["segment"] == s]["rfm_score"].values for s in seg_order]
    vp = ax.violinplot(seg_rfm, positions=range(len(seg_order)), showmedians=True, showextrema=True)
    for i, (pc, seg) in enumerate(zip(vp["bodies"], seg_order)):
        pc.set_facecolor(SEG_COLORS[seg]); pc.set_alpha(0.8)
    vp["cmedians"].set_color("white"); vp["cmedians"].set_linewidth(2)
    ax.set_xticks(range(len(seg_order)))
    ax.set_xticklabels(seg_order, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("RFM Score (Weighted)", fontsize=10)
    ax.set_title("RFM Score Distribution by Segment", fontsize=13, fontweight="bold", pad=15)
    ax.set_ylim(0.5, 5.5)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/03_rfm_score_violin.png", bbox_inches="tight")
    plt.close()
    print("    03_rfm_score_violin.png")

    # ── Chart 4: Recency vs Monetary scatter (by segment)
    fig, ax = plt.subplots(figsize=(11, 7))
    for seg in seg_order:
        sub = rfm_full[rfm_full["segment"] == seg]
        ax.scatter(sub["recency_days"], sub["monetary"],
                   color=SEG_COLORS[seg], alpha=0.45, s=18, label=seg)
    ax.set_xlabel("Recency (Days since last order)", fontsize=10)
    ax.set_ylabel("Monetary Value ($)", fontsize=10)
    ax.set_title("Recency vs Monetary Value — Coloured by Segment", fontsize=13, fontweight="bold", pad=15)
    ax.legend(loc="upper right", fontsize=8, frameon=True, framealpha=0.9)
    ax.set_yscale("log")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/04_recency_vs_monetary.png", bbox_inches="tight")
    plt.close()
    print("    04_recency_vs_monetary.png")

    # ── Chart 5: Monthly revenue trend
    fig, ax = plt.subplots(figsize=(14, 5))
    months_str = monthly_rev["year_month"].astype(str)
    ax.fill_between(range(len(months_str)), monthly_rev["revenue"],
                    alpha=0.15, color="#1F4E79")
    ax.plot(range(len(months_str)), monthly_rev["revenue"],
            color="#1F4E79", linewidth=2)
    ax.set_xticks(range(0, len(months_str), 6))
    ax.set_xticklabels(months_str[::6], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Monthly Revenue ($)", fontsize=10)
    ax.set_title("Monthly Revenue Trend — ShopNest 2020–2024", fontsize=13, fontweight="bold", pad=15)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x/1e3:.0f}K"))
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/05_monthly_revenue_trend.png", bbox_inches="tight")
    plt.close()
    print("    05_monthly_revenue_trend.png")

    # ── Chart 6: RFM heatmap (avg monetary by R×F score)
    pivot = rfm_full.pivot_table(values="monetary", index="r_score",
                                  columns="f_score", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, cbar_kws={"label":"Avg Monetary ($)"})
    ax.set_xlabel("Frequency Score (F)", fontsize=10)
    ax.set_ylabel("Recency Score (R)", fontsize=10)
    ax.set_title("Average Monetary Value by R × F Score", fontsize=12, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/06_rfm_heatmap.png", bbox_inches="tight")
    plt.close()
    print("    06_rfm_heatmap.png")

except ImportError as e:
    print("    Install with: pip install matplotlib seaborn")




[12] Generating charts …
    01_segment_customer_count.png
    02_revenue_share_donut.png
    03_rfm_score_violin.png
    04_recency_vs_monetary.png
    05_monthly_revenue_trend.png
    06_rfm_heatmap.png


In [64]:
# FINAL SUMMARY
print("  ANALYSIS COMPLETE")
print("=" * 65)
print(f"  Customers analysed : {len(rfm_full):,}")
print(f"  Total Revenue      : ${total_rev:,.2f}")
print(f"  Segments found     : {rfm_full['segment'].nunique()}")
print(f"  Output files saved : {OUTPUT_DIR}/")


  ANALYSIS COMPLETE
  Customers analysed : 4,772
  Total Revenue      : $32,117,809.72
  Segments found     : 8
  Output files saved : rfm_output/


In [66]:
import shutil

shutil.make_archive("rfm_output", "zip", OUTPUT_DIR)

'/content/rfm_output.zip'

In [67]:
from google.colab import files

files.download("rfm_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>